In [0]:
from pyspark.sql.functions import col, trim, upper
from pyspark.sql.types import DoubleType, LongType, TimestampType, StringType

# Define parameters via widgets
dbutils.widgets.text("catalog", "dbr_dev", "1. Catalog Name")
dbutils.widgets.text("bronze_schema", "valeriimatviiv_bronze", "2. Bronze Schema")
dbutils.widgets.text("silver_schema", "valeriimatviiv_silver", "3. Silver Schema")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")

bronze_table = f"{catalog}.{bronze_schema}.nasdaq_price_bronze"
silver_table = f"{catalog}.{silver_schema}.nasdaq_price_silver"


In [0]:
df_bronze = spark.read.table(bronze_table)

# Clean, cast, and deduplicate
df_silver = (
    df_bronze
    .withColumn("Symbol", upper(trim(col("Symbol").cast(StringType()))))
    .withColumn("TradeTimestamp", col("TradeTimestamp").cast(TimestampType()))
    .withColumn("Open", col("Open").cast(DoubleType()))
    .withColumn("High", col("High").cast(DoubleType()))
    .withColumn("Low", col("Low").cast(DoubleType()))
    .withColumn("Close", col("Close").cast(DoubleType()))
    .withColumn("Volume", col("Volume").cast(LongType()))
    .filter(col("Symbol").isNotNull() & col("TradeTimestamp").isNotNull())
    .dropDuplicates(["Symbol", "TradeTimestamp"])
    .select("Symbol", "TradeTimestamp", "Open", "High", "Low", "Close", "Volume")
)

# Overwrite Silver Delta Table
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

# print(f"Successfully processed {df_silver.count()} clean price rows into Silver table: {silver_table}")

In [0]:
# df_verify_price = spark.table(f"{catalog}.{silver_schema}.nasdaq_price_silver")

# print(f"Total Silver Price Rows: {df_verify_price.count()}")
# print("Schema:")
# df_verify_price.printSchema()
# display(df_verify_price.limit(5))